# Prototype Model Evaluation

This notebook reloads the prototype XGBoost model, scores validation and test splits, and reports ranking metrics at the track level.

Notes:
- Metrics are computed after ranking target countries within each track.
- The default configuration samples whole tracks for faster iteration.
- Set `VAL_MAX_TRACKS` or `TEST_MAX_TRACKS` to `None` if you want to run on the full split and have enough memory.


In [2]:
from pathlib import Path
import json
import warnings

import duckdb
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score

try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError("Install xgboost first, e.g. `pip install -r requirements.txt`.") from exc

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "datasets").exists() and (candidate / "requirements.txt").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. Expected a parent containing 'datasets/' and 'requirements.txt'."
    )

ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = ROOT / "datasets" / "processed" / "v3_features"
MODEL_DIR = ROOT / "artifacts" / "models" / "xgboost_prototype"
EVAL_DIR = ROOT / "artifacts" / "evaluations" / "xgboost_prototype"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "xgb_classifier.json"
SUMMARY_PATH = MODEL_DIR / "training_summary.json"
VAL_PATH = DATA_DIR / "val.parquet"
TEST_PATH = DATA_DIR / "test.parquet"

assert MODEL_PATH.exists(), f"Missing model artifact: {MODEL_PATH}"
assert SUMMARY_PATH.exists(), f"Missing metadata artifact: {SUMMARY_PATH}"
assert VAL_PATH.exists(), f"Missing validation split: {VAL_PATH}"
assert TEST_PATH.exists(), f"Missing test split: {TEST_PATH}"

TOP_K = 5
VAL_MAX_TRACKS = 10000
TEST_MAX_TRACKS = 10000


In [3]:
con = duckdb.connect()

def load_split(path: Path, max_tracks: int | None = None) -> pd.DataFrame:
    parquet_path = path.as_posix()
    if max_tracks is None:
        query = f"SELECT * FROM read_parquet('{parquet_path}')"
    else:
        query = f"""
            WITH sampled_tracks AS (
                SELECT track_id
                FROM read_parquet('{parquet_path}')
                GROUP BY track_id
                ORDER BY hash(track_id)
                LIMIT {int(max_tracks)}
            )
            SELECT d.*
            FROM read_parquet('{parquet_path}') d
            JOIN sampled_tracks st USING (track_id)
        """
    return con.execute(query).fetchdf()

def ranking_metrics(scored_df: pd.DataFrame, k: int = 5) -> tuple[dict, pd.DataFrame]:
    rows = []
    for track_id, group in scored_df.groupby("track_id", sort=False):
        group = group.sort_values("score", ascending=False).reset_index(drop=True)
        labels = group["did_enter_within_60d"].to_numpy(dtype=int)
        positives = int(labels.sum())
        top = group.head(k)
        top_labels = top["did_enter_within_60d"].to_numpy(dtype=int)
        hits = int(top_labels.sum())

        precision = hits / k
        recall = hits / positives if positives else np.nan
        hit_rate = float(hits > 0) if positives else np.nan

        discounts = np.log2(np.arange(2, len(top_labels) + 2))
        dcg = float(((2 ** top_labels - 1) / discounts).sum())
        ideal = np.sort(labels)[::-1][: len(top_labels)]
        idcg = float(((2 ** ideal - 1) / np.log2(np.arange(2, len(ideal) + 2))).sum())
        ndcg = dcg / idcg if idcg > 0 else np.nan

        ap_accum = 0.0
        running_hits = 0
        for rank, rel in enumerate(top_labels, start=1):
            if rel:
                running_hits += 1
                ap_accum += running_hits / rank
        map_k = ap_accum / min(positives, k) if positives else np.nan

        rows.append(
            {
                "track_id": track_id,
                "positives": positives,
                f"precision@{k}": precision,
                f"recall@{k}": recall,
                f"hit_rate@{k}": hit_rate,
                f"ndcg@{k}": ndcg,
                f"map@{k}": map_k,
            }
        )

    metric_df = pd.DataFrame(rows)
    positive_mask = metric_df["positives"] > 0
    summary = {
        "tracks": int(metric_df.shape[0]),
        "positive_tracks": int(positive_mask.sum()),
        f"precision@{k}": float(metric_df[f"precision@{k}"].mean()),
        f"recall@{k}": float(metric_df.loc[positive_mask, f"recall@{k}"].mean()) if positive_mask.any() else None,
        f"hit_rate@{k}": float(metric_df.loc[positive_mask, f"hit_rate@{k}"].mean()) if positive_mask.any() else None,
        f"ndcg@{k}": float(metric_df.loc[positive_mask, f"ndcg@{k}"].mean()) if positive_mask.any() else None,
        f"map@{k}": float(metric_df.loc[positive_mask, f"map@{k}"].mean()) if positive_mask.any() else None,
        "mean_future_countries_per_track": float(metric_df["positives"].mean()),
    }
    return summary, metric_df

def binary_metrics(y_true: pd.Series, y_score: np.ndarray) -> dict:
    return {
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "average_precision": float(average_precision_score(y_true, y_score)),
        "log_loss": float(log_loss(y_true, y_score, labels=[0, 1])),
    }

def score_split(model: xgb.XGBClassifier, df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series) -> pd.DataFrame:
    X = df[feature_cols].fillna(fill_values)
    scores = model.predict_proba(X)[:, 1]
    scored = df[["track_id", "observation_time", "target_country", "did_enter_within_60d", "days_to_entry"]].copy()
    scored["score"] = scores
    return scored


In [4]:
with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    training_summary = json.load(f)

feature_cols = training_summary["feature_cols"]
fill_values = pd.Series(training_summary["train_fill_values"], dtype=float)

model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)

val_df = load_split(VAL_PATH, VAL_MAX_TRACKS)
test_df = load_split(TEST_PATH, TEST_MAX_TRACKS)

print(f"Loaded validation rows: {len(val_df):,} | tracks: {val_df['track_id'].nunique():,}")
print(f"Loaded test rows:       {len(test_df):,} | tracks: {test_df['track_id'].nunique():,}")


Loaded validation rows: 636,230 | tracks: 10,000
Loaded test rows:       638,276 | tracks: 10,000


In [5]:
val_scored = score_split(model, val_df, feature_cols, fill_values)
test_scored = score_split(model, test_df, feature_cols, fill_values)

val_binary = binary_metrics(val_scored["did_enter_within_60d"], val_scored["score"].to_numpy())
test_binary = binary_metrics(test_scored["did_enter_within_60d"], test_scored["score"].to_numpy())

val_ranking, val_track_metrics = ranking_metrics(val_scored, k=TOP_K)
test_ranking, test_track_metrics = ranking_metrics(test_scored, k=TOP_K)

evaluation_summary = {
    "config": {
        "top_k": TOP_K,
        "val_max_tracks": VAL_MAX_TRACKS,
        "test_max_tracks": TEST_MAX_TRACKS,
    },
    "validation": {
        "binary": val_binary,
        "ranking": val_ranking,
    },
    "test": {
        "binary": test_binary,
        "ranking": test_ranking,
    },
}

pd.DataFrame(
    {
        "validation": {**val_binary, **val_ranking},
        "test": {**test_binary, **test_ranking},
    }
).T


,roc_auc,average_precision,log_loss,tracks,positive_tracks,precision@5,recall@5,hit_rate@5,ndcg@5,map@5,mean_future_countries_per_track
validation,0.954251,0.233573,0.158427,10000.0,811.0,0.02752,0.598974,0.831073,0.639056,0.578129,0.4038
test,0.942934,0.287481,0.160096,10000.0,871.0,0.03026,0.616850,0.833525,0.654178,0.594983,0.4797


In [6]:
country_breakdown = (
    test_scored.groupby("target_country")
    .agg(
        rows=("score", "size"),
        positives=("did_enter_within_60d", "sum"),
        avg_score=("score", "mean"),
    )
    .sort_values(["positives", "avg_score"], ascending=[False, False])
)

positive_test_tracks = test_scored.groupby("track_id")["did_enter_within_60d"].sum()
sample_track_id = positive_test_tracks[positive_test_tracks > 0].index[0]
sample_track = (
    test_scored[test_scored["track_id"] == sample_track_id]
    .sort_values("score", ascending=False)
    .head(10)
)

summary_path = EVAL_DIR / "evaluation_summary.json"
val_preds_path = EVAL_DIR / "val_predictions.parquet"
test_preds_path = EVAL_DIR / "test_predictions.parquet"
track_metrics_path = EVAL_DIR / "test_track_metrics.parquet"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, indent=2)
con.register("val_scored_df", val_scored)
con.register("test_scored_df", test_scored)
con.register("test_track_metrics_df", test_track_metrics)
con.execute(f"COPY val_scored_df TO '{val_preds_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY test_scored_df TO '{test_preds_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY test_track_metrics_df TO '{track_metrics_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.unregister("val_scored_df")
con.unregister("test_scored_df")
con.unregister("test_track_metrics_df")

print(json.dumps(evaluation_summary, indent=2))
print(f"Saved evaluation summary to: {summary_path}")
print(f"Saved validation predictions to: {val_preds_path}")
print(f"Saved test predictions to: {test_preds_path}")
print(f"Saved test track metrics to: {track_metrics_path}")

print("\nTop countries by positive count in the scored test sample:")
display(country_breakdown.head(15))

print(f"\nSample track with future entries: {sample_track_id}")
sample_track


{
  "config": {
    "top_k": 5,
    "val_max_tracks": 10000,
    "test_max_tracks": 10000
  },
  "validation": {
    "binary": {
      "roc_auc": 0.9542507400887033,
      "average_precision": 0.23357310088500333,
      "log_loss": 0.15842729911801587
    },
    "ranking": {
      "tracks": 10000,
      "positive_tracks": 811,
      "precision@5": 0.027520000000000006,
      "recall@5": 0.5989744937773717,
      "hit_rate@5": 0.8310727496917386,
      "ndcg@5": 0.6390557231427955,
      "map@5": 0.5781291957802438,
      "mean_future_countries_per_track": 0.4038
    }
  },
  "test": {
    "binary": {
      "roc_auc": 0.9429337962054449,
      "average_precision": 0.28748095189336303,
      "log_loss": 0.1600961986920159
    },
    "ranking": {
      "tracks": 10000,
      "positive_tracks": 871,
      "precision@5": 0.030260000000000002,
      "recall@5": 0.6168497646159622,
      "hit_rate@5": 0.8335246842709529,
      "ndcg@5": 0.6541776434064054,
      "map@5": 0.5949834162520728,
 

,rows,positives,avg_score
target_country,,,
Switzerland,9326,164,0.210011
Belgium,9644,137,0.194342
Portugal,9744,102,0.138613
Lithuania,9525,100,0.132073
Austria,9423,99,0.154628
United Arab Emirates,9695,98,0.123701
Paraguay,9878,97,0.112063
Estonia,9736,91,0.123460
Sweden,9402,90,0.134285



Sample track with future entries: 00Blm7zeNqgYLPtW6zg8cj


,track_id,observation_time,target_country,did_enter_within_60d,days_to_entry,score
390559,00Blm7zeNqgYLPtW6zg8cj,2021-11-05,Luxembourg,1,14,0.972760
60026,00Blm7zeNqgYLPtW6zg8cj,2021-11-05,Paraguay,0,<NA>,0.967785
475369,00Blm7zeNqgYLPtW6zg8cj,2021-11-05,Indonesia,1,3,0.966542
10317,00Blm7zeNqgYLPtW6zg8cj,2021-11-05,Japan,1,3,0.965688
252247,00Blm7zeNqgYLPtW6zg8cj,2021-11-05,Uruguay,1,3,0.951704
492379,00Blm7zeNqgYLPtW6zg8cj,2021-11-05,Chile,1,3,0.948884
60032,00Blm7zeNqgYLPtW6zg8cj,2021-11-05,Peru,0,<NA>,0.947226
369180,00Blm7zeNqgYLPtW6zg8cj,2021-11-05,Argentina,0,<NA>,0.942256
511250,00Blm7zeNqgYLPtW6zg8cj,2021-11-05,Andorra,0,<NA>,0.628367
